In [1]:
import csv
import gzip
import json
import math
from collections import OrderedDict
from dataclasses import dataclass
from typing import Union, Tuple, List, Iterable, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr
from torch import nn, Tensor
from torch.nn.parameter import Parameter
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer

torch.manual_seed(0)
np.random.seed(0)

In [2]:
@dataclass
class BertConfig:
    context_size: int = 512
    vocab_size: int = 30522
    num_hidden_layers: int = 12
    hidden_size: int = 768
    num_attention_heads: int = 12
    dropout_prob: float = 0.1


model_id = "prajjwal1/bert-tiny"

**TODO**

1. Check what the different code pieces are doing
    - Inspect the shapes
1. Try on some actual inputs
1. Try make it faster?
    - Maybe later?
1. Check that other model sizes work

In [3]:
class BertEmbeddings(nn.Module):

    def __init__(self, config: BertConfig):
        super().__init__()
        self.config = config

        self.word_embeddings = nn.Embedding(self.config.vocab_size, self.config.hidden_size, padding_idx=0)
        self.position_embeddings = nn.Embedding(self.config.context_size, self.config.hidden_size)
        self.token_type_embeddings = nn.Embedding(2, self.config.hidden_size)
        self.LayerNorm = nn.LayerNorm(self.config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(self.config.dropout_prob)

        self.register_buffer("position_ids", torch.arange(config.context_size).expand((1, -1)), persistent=False)
        self.register_buffer(
            "token_type_ids", torch.zeros(self.position_ids.size(), dtype=torch.long), persistent=False
        )

    def forward(self, input_ids, position_ids, token_type_ids):
        B, T = input_ids.shape

        if position_ids is None:
            position_ids = self.position_ids[:, :T]

        if token_type_ids is None:
            buffered_token_type_ids = self.token_type_ids[:, :T]
            buffered_token_type_ids_expanded = buffered_token_type_ids.expand(B, T)
            token_type_ids = buffered_token_type_ids_expanded

        embeddings = (
            self.word_embeddings(input_ids)
            + self.token_type_embeddings(token_type_ids)
            + self.position_embeddings(position_ids)
        )
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings


class BertSelfAttention(nn.Module):

    def __init__(self, config: BertConfig):
        super().__init__()
        self.config = config

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = config.hidden_size // config.num_attention_heads
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.dropout_prob)

    def transpose_for_scores(self, x: Tensor) -> Tensor:
        new_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states: Tensor, attention_mask: Tensor | None = None):
        B, T, _ = hidden_states.size()

        query_layer = self.query(hidden_states)
        key_layer = self.key(hidden_states)
        value_layer = self.value(hidden_states)

        query_layer = self.transpose_for_scores(query_layer)
        key_layer = self.transpose_for_scores(key_layer)
        value_layer = self.transpose_for_scores(value_layer)

        attn_output = torch.nn.functional.scaled_dot_product_attention(
            query_layer,
            key_layer,
            value_layer,
            attn_mask=attention_mask,
            dropout_p=self.dropout_prob if self.training else 0.0,
            is_causal=False,
        )

        attn_output = attn_output.transpose(1, 2)
        attn_output = attn_output.reshape(B, T, self.all_head_size)
        return attn_output


class BertSelfOutput(nn.Module):

    def __init__(self, config: BertConfig):
        super().__init__()
        self.config = config

        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(config.dropout_prob)

    def forward(self, hidden_states: Tensor, input_tensor: Tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


class BertAttention(nn.Module):

    def __init__(self, config: BertConfig):
        super().__init__()
        self.config = config

        self.self = BertSelfAttention(config)
        self.output = BertSelfOutput(config)

    def forward(self, hidden_states: Tensor, attention_mask: Tensor | None = None):
        self_output = self.self(hidden_states, attention_mask)
        attention_output = self.output(self_output, hidden_states)
        return attention_output


class BertIntermediate(nn.Module):

    def __init__(self, config: BertConfig):
        super().__init__()
        self.config = config

        self.dense = nn.Linear(config.hidden_size, 4 * config.hidden_size)
        self.intermediate_act_fn = nn.GELU()

    def forward(self, hidden_states: Tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states


class BertOutput(nn.Module):

    def __init__(self, config: BertConfig):
        super().__init__()
        self.config = config

        self.dense = nn.Linear(4 * config.hidden_size, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(config.dropout_prob)

    def forward(self, hidden_states: Tensor, input_tensor: Tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


class BertLayer(nn.Module):

    def __init__(self, config: BertConfig):
        super().__init__()
        self.config = config

        self.attention = BertAttention(config)
        self.intermediate = BertIntermediate(config)
        self.output = BertOutput(config)

    def forward(self, hidden_states: Tensor, attention_mask: Tensor | None = None):
        attention_output = self.attention(hidden_states, attention_mask)
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output


class BertPooler(nn.Module):

    def __init__(self, config: BertConfig):
        super().__init__()
        self.config = config

        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.activation = nn.Tanh()

    def forward(self, hidden_states: Tensor):
        pooled_output = self.dense(hidden_states[:, 0])
        return self.activation(pooled_output)


class BertEncoder(nn.Module):

    def __init__(self, config: BertConfig):
        super().__init__()
        self.config = config

        self.layer = nn.ModuleList([BertLayer(config) for _ in range(config.num_hidden_layers)])

    def forward(self, hidden_states: Tensor, attention_mask: Tensor | None = None):
        for layer_module in self.layer:
            hidden_states = layer_module(hidden_states, attention_mask)
        return hidden_states


class Bert(nn.Module):

    def __init__(self, config: BertConfig):
        super().__init__()
        self.config = config

        self.embeddings = BertEmbeddings(config)
        self.encoder = BertEncoder(config)
        self.pooler = BertPooler(config)

    def forward(self, input_ids: Tensor, attention_mask: Tensor | None = None, token_type_ids: Tensor | None = None):
        B, T = input_ids.shape
        assert T <= self.config.context_size, f"Input length {T} exceeds context size {self.config.context_size}"

        x = self.embeddings(input_ids, None, token_type_ids)

        if attention_mask is None:
            attention_mask = torch.ones((B, T), device=input_ids.device)

        inverted_mask = 1.0 - attention_mask[:, None, None, :].expand(B, 1, T, T).to(x.dtype)

        inverted_mask = inverted_mask.masked_fill(inverted_mask.to(torch.bool), torch.finfo(x.dtype).min)

        x = self.encoder(x, inverted_mask)

        return x, self.pooler(x)

    @classmethod
    def from_pretrained(cls, model_id: str):
        config_args = {
            "hidden_size": 128,
            "num_attention_heads": 2,
            "num_hidden_layers": 2,
            "vocab_size": 30522,  # example vocab size
            "context_size": 512,  # example context size
            "dropout_prob": 0.1,  # example dropout probability
        }
        config = BertConfig(**config_args)

        model = Bert(config)
        model_hf = AutoModel.from_pretrained(model_id)

        model_dict = model.state_dict()
        model_dict_hf = model_hf.state_dict()

        for name, param in model_dict_hf.items():
            if name in model_dict:
                model_dict[name].copy_(param)
            else:
                raise KeyError(f"{name} not found in custom model state_dict")

        model.load_state_dict(model_dict)

        return model

In [4]:
model = Bert.from_pretrained(model_id).eval()

model_hf = AutoModel.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

/Users/augustasm/miniconda3/envs/website/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
sample_sentence = "An example use of pretrained BERT with transformers library to encode a sentence"
tokenized_sample_sentence = tokenizer(sample_sentence, return_tensors="pt", padding="max_length", max_length=512)

with torch.no_grad():  # Added torch.no_grad() to avoid OOM errors
    output, _ = model(
        input_ids=tokenized_sample_sentence["input_ids"],
        attention_mask=tokenized_sample_sentence["attention_mask"],
    )

    output_hf = model_hf(
        input_ids=tokenized_sample_sentence["input_ids"],
        attention_mask=tokenized_sample_sentence["attention_mask"],
    ).last_hidden_state

output.shape, output_hf.shape

(torch.Size([1, 512, 128]), torch.Size([1, 512, 128]))

In [6]:
print((output - output_hf).pow(2).mean().sqrt().item())

assert torch.allclose(output, output_hf, atol=1e-5)

0.0


In [8]:
def compare_layer_outputs(model, model_hf, tokenized_sample_sentence):
    input_ids = tokenized_sample_sentence["input_ids"]
    attention_mask = tokenized_sample_sentence["attention_mask"].bool()  # Ensure attention_mask is boolean
    token_type_ids = tokenized_sample_sentence.get("token_type_ids", torch.zeros_like(input_ids))

    with torch.no_grad():
        x = model.embeddings(
            input_ids=input_ids,
            position_ids=torch.arange(input_ids.size(1), dtype=torch.long, device=input_ids.device),
            token_type_ids=token_type_ids,
        )
        x_hf = model_hf.embeddings(input_ids=input_ids, token_type_ids=token_type_ids)

        print("Embeddings: ", torch.allclose(x, x_hf, atol=1e-7))
        print("Embeddings: ", (x - x_hf).pow(2).mean().sqrt().item())

        for i, layer in enumerate(model.encoder.layer):
            x = layer.attention(x, attention_mask)
            x_hf = model_hf.encoder.layer[i].attention(hidden_states=x_hf, attention_mask=attention_mask)[0]
            print(f"Layer {i} attention output: ", torch.allclose(x, x_hf))
            print(f"Layer {i} attention output: ", (x - x_hf).pow(2).mean().sqrt().item())

            intermediate_output = layer.intermediate(x)
            intermediate_output_hf = model_hf.encoder.layer[i].intermediate(x_hf)
            print(
                f"Layer {i} intermediate output: ",
                torch.allclose(intermediate_output, intermediate_output_hf),
            )
            print(
                f"Layer {i} intermediate output: ",
                (intermediate_output - intermediate_output_hf).pow(2).mean().sqrt().item(),
            )

            layer_output = layer.output(intermediate_output, x)
            layer_output_hf = model_hf.encoder.layer[i].output(intermediate_output_hf, x_hf)
            print(f"Layer {i} output: ", torch.allclose(layer_output, layer_output_hf))
            print(f"Layer {i} output: ", (layer_output - layer_output_hf).pow(2).mean().sqrt().item())

            x = layer_output
            x_hf = layer_output_hf

        pooled_output = model.pooler(x)
        pooled_output_hf = model_hf.pooler(x_hf)
        print("Pooled output: ", torch.allclose(pooled_output, pooled_output_hf))
        print("Pooled output: ", (pooled_output - pooled_output_hf).pow(2).mean().sqrt().item())


# Run the comparison
compare_layer_outputs(model, model_hf, tokenized_sample_sentence)

Embeddings:  True
Embeddings:  0.0
Layer 0 attention output:  True
Layer 0 attention output:  0.0
Layer 0 intermediate output:  True
Layer 0 intermediate output:  0.0
Layer 0 output:  True
Layer 0 output:  0.0
Layer 1 attention output:  True
Layer 1 attention output:  0.0
Layer 1 intermediate output:  True
Layer 1 intermediate output:  0.0
Layer 1 output:  True
Layer 1 output:  0.0
Pooled output:  True
Pooled output:  0.0
